# Evaluation

<div style="text-align: justify">

The following notebook evaluates both the <b>Autoencoder (AE)</b> and <b>Variational Autoencoder (VAE)</b> for the <b>Tau Anomaly Detection</b> analysis. Trained checkpoints are loaded, per-event anomaly scores are computed, and evaluation metrics (ROC AUC, SIC) are produced. The notebook includes side-by-side comparison of both models and VAE-specific latent space diagnostics.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration for AE and VAE |
| DataModule | `datamodule.AnomalyDataModule` | Read mc.parquet, split bkg/sig, fit scaler, build dataloaders |
| Load Models | `ae.Autoencoder`, `vae.VariationalAutoencoder` | Load trained checkpoints |
| Scores | `anomaly.reconstruction_error` | Compute per-event anomaly scores on predict set |
| Metrics | `evaluation.compute_metrics` | ROC AUC, SIC, per-origin ROC |
| Plots | `plots` | Reconstruction error, ROC, SIC, latent diagnostics, comparison |

The same pipeline is available as a CLI via `uv run python run.py stage=evaluate model=ae|vae`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
* [scikit-learn](https://scikit-learn.org/)

Visualization:
* [matplotlib](https://matplotlib.org/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration for both AE and VAE models.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=ae"])
cfg_vae = compose(config_name="config", overrides=["model=vae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
ae_plots_dir = path / output_paths["plots_dir"] / "ae_evaluation"
vae_plots_dir = path / output_paths["plots_dir"] / "vae_evaluation"

ae_plots_dir.mkdir(parents=True, exist_ok=True)
vae_plots_dir.mkdir(parents=True, exist_ok=True)

## DataModule

Setting up the `AnomalyDataModule`. It reads the MC parquet, separates background from signal, fits a scaler on the background training split, and builds the predict set (background validation + all signal).

In [ ]:
import lightning as L

L.seed_everything(cfg.seed, workers=True)

In [ ]:
from src.models.datamodule import AnomalyDataModule

background_origins = {
    s["id"]
    for s in cfg.samples.background.samples
    if s["id"] not in set(cfg.samples.background.get("exclude", []))
}
print(f"Background origins: {background_origins}")

dm = AnomalyDataModule(
    mc_path=str(dataframes_dir / "mc.parquet"),
    background_origins=background_origins,
    normalization=cfg.model.normalization,
    val_fraction=cfg.pipeline.val_fraction,
    batch_size=cfg.model.batch_size,
    seed=cfg.seed,
)
dm.setup()
print(f"Features: {dm.n_features}")
print(f"Feature names: {dm.feature_names_}")

Gathering original features from the predict set (shared by both models).

In [ ]:
import torch

x_orig = torch.cat([batch[0] for batch in dm.predict_dataloader()])
print(
    f"Predict set: {len(x_orig):,} events "
    f"({int((dm.predict_labels == 0).sum()):,} bkg, "
    f"{int((dm.predict_labels == 1).sum()):,} sig)"
)

## AE Evaluation

### Load Checkpoint

Loading the trained Autoencoder from checkpoint.

In [ ]:
from omegaconf import OmegaConf

from src.models.ae import Autoencoder
from src.models.config import AEConfig

ae_params = dict(OmegaConf.to_container(cfg.model, resolve=True))
ae_cfg = AEConfig(**ae_params)
ae_model = Autoencoder.load_from_checkpoint(
    models_dir / "ae.ckpt", cfg=ae_cfg, n_features=dm.n_features
)
ae_model.eval()

n_params = sum(p.numel() for p in ae_model.parameters())
print(f"Autoencoder: {n_params:,} parameters")

### Anomaly Scores

Computing per-event reconstruction error on the predict set.

In [ ]:
import numpy as np

from src.models.anomaly import (
    build_scores_frame,
    compute_threshold,
    per_feature_error,
    reconstruction_error,
)

ae_x_hat_list = []
with torch.no_grad():
    for batch in dm.predict_dataloader():
        x, _w = batch
        x_hat = ae_model(x)
        ae_x_hat_list.append(x_hat)

ae_x_hat = torch.cat(ae_x_hat_list)
ae_scores = reconstruction_error(x_orig, ae_x_hat).numpy()

print(f"AE scores: {len(ae_scores):,} events")
print(f"  Background mean: {ae_scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {ae_scores[dm.predict_labels == 1].mean():.6f}")

### Threshold

In [ ]:
ae_bkg_scores = ae_scores[dm.predict_labels == 0]
ae_sig_scores = ae_scores[dm.predict_labels == 1]

ae_threshold = compute_threshold(
    ae_bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {ae_threshold:.6f}")
print(f"  Background flagged: {(ae_bkg_scores > ae_threshold).sum():,} / {len(ae_bkg_scores):,}")
print(f"  Signal flagged:     {(ae_sig_scores > ae_threshold).sum():,} / {len(ae_sig_scores):,}")

### Metrics

In [ ]:
from src.models.evaluation import compute_metrics, compute_roc_curve, compute_sic_curve

ae_scores_df = build_scores_frame(ae_scores, dm.predict_labels, dm.predict_origins)
ae_metrics = compute_metrics(
    labels=dm.predict_labels,
    scores=ae_scores,
    scores_df=ae_scores_df,
)
ae_metrics["threshold"] = ae_threshold
ae_metrics["n_background"] = int((dm.predict_labels == 0).sum())
ae_metrics["n_signal"] = int((dm.predict_labels == 1).sum())

print(f"AE ROC AUC: {ae_metrics['roc_auc']:.4f}")
print(f"AE max SIC: {ae_metrics['max_sic']:.4f}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
from src.models.plots import plot_reconstruction_error

fig = plot_reconstruction_error(
    ae_bkg_scores, ae_sig_scores, threshold=ae_threshold, title="AE Reconstruction Error"
)
fig.savefig(ae_plots_dir / "reconstruction_error.png", dpi=150, bbox_inches="tight")

### ROC Curve

In [ ]:
from src.models.plots import plot_roc_curve

ae_fpr, ae_tpr, _ = compute_roc_curve(dm.predict_labels, ae_scores)
fig = plot_roc_curve(ae_fpr, ae_tpr, auc=ae_metrics["roc_auc"], title="AE ROC Curve")
fig.savefig(ae_plots_dir / "roc_curve.png", dpi=150, bbox_inches="tight")

### SIC Curve

Significance Improvement Characteristic: signal efficiency divided by the square root of background efficiency.

In [ ]:
from src.models.plots import plot_sic_curve

ae_sic = compute_sic_curve(ae_fpr, ae_tpr)
fig = plot_sic_curve(ae_fpr, ae_tpr, ae_sic, title="AE Significance Improvement")
fig.savefig(ae_plots_dir / "sic_curve.png", dpi=150, bbox_inches="tight")

### Per-Origin ROC

ROC AUC computed separately for each signal mass point.

In [ ]:
from src.models.plots import plot_roc_per_origin

if ae_metrics.get("roc_per_origin"):
    fig = plot_roc_per_origin(
        ae_metrics["roc_per_origin"], title="AE ROC AUC per Signal Origin"
    )
    fig.savefig(ae_plots_dir / "roc_per_origin.png", dpi=150, bbox_inches="tight")
    for origin, auc in sorted(ae_metrics["roc_per_origin"].items()):
        print(f"  {origin}: {auc:.4f}")

### Per-Feature Importance

Mean squared reconstruction error per feature, identifying which features contribute most to the anomaly score.

In [ ]:
from src.models.plots import plot_per_feature_importance

ae_feat_errors = per_feature_error(x_orig, ae_x_hat).numpy()
ae_mean_feat_errors = ae_feat_errors.mean(axis=0)
fig = plot_per_feature_importance(
    ae_mean_feat_errors, dm.feature_names_, title="AE Per-Feature Reconstruction Error"
)
fig.savefig(ae_plots_dir / "per_feature_importance.png", dpi=150, bbox_inches="tight")

## VAE Evaluation

### Load Checkpoint

Loading the trained Variational Autoencoder from checkpoint.

In [ ]:
from src.models.config import VAEConfig
from src.models.vae import VariationalAutoencoder

vae_params = dict(OmegaConf.to_container(cfg_vae.model, resolve=True))
vae_cfg = VAEConfig(**vae_params)
vae_model = VariationalAutoencoder.load_from_checkpoint(
    models_dir / "vae.ckpt", cfg=vae_cfg, n_features=dm.n_features
)
vae_model.eval()

n_params = sum(p.numel() for p in vae_model.parameters())
print(f"VAE: {n_params:,} parameters")

### Anomaly Scores

Computing per-event reconstruction error on the predict set. Also extracting `mu` and `logvar` for latent space analysis.

In [ ]:
vae_x_hat_list = []
vae_mu_list, vae_logvar_list = [], []

with torch.no_grad():
    for batch in dm.predict_dataloader():
        x, _w = batch
        x_hat, mu, logvar = vae_model(x)
        vae_x_hat_list.append(x_hat)
        vae_mu_list.append(mu)
        vae_logvar_list.append(logvar)

vae_x_hat = torch.cat(vae_x_hat_list)
mu_np = torch.cat(vae_mu_list).numpy()
logvar_np = torch.cat(vae_logvar_list).numpy()
vae_scores = reconstruction_error(x_orig, vae_x_hat).numpy()

print(f"VAE scores: {len(vae_scores):,} events")
print(f"  Background mean: {vae_scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {vae_scores[dm.predict_labels == 1].mean():.6f}")
print(f"  Latent dim: {mu_np.shape[1]}")

### Threshold

In [ ]:
vae_bkg_scores = vae_scores[dm.predict_labels == 0]
vae_sig_scores = vae_scores[dm.predict_labels == 1]

vae_threshold = compute_threshold(
    vae_bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {vae_threshold:.6f}")
print(f"  Background flagged: {(vae_bkg_scores > vae_threshold).sum():,} / {len(vae_bkg_scores):,}")
print(f"  Signal flagged:     {(vae_sig_scores > vae_threshold).sum():,} / {len(vae_sig_scores):,}")

### Metrics

In [ ]:
vae_scores_df = build_scores_frame(vae_scores, dm.predict_labels, dm.predict_origins)
vae_metrics = compute_metrics(
    labels=dm.predict_labels,
    scores=vae_scores,
    scores_df=vae_scores_df,
)
vae_metrics["threshold"] = vae_threshold
vae_metrics["n_background"] = int((dm.predict_labels == 0).sum())
vae_metrics["n_signal"] = int((dm.predict_labels == 1).sum())

print(f"VAE ROC AUC: {vae_metrics['roc_auc']:.4f}")
print(f"VAE max SIC: {vae_metrics['max_sic']:.4f}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
fig = plot_reconstruction_error(
    vae_bkg_scores, vae_sig_scores, threshold=vae_threshold, title="VAE Reconstruction Error"
)
fig.savefig(vae_plots_dir / "reconstruction_error.png", dpi=150, bbox_inches="tight")

### ROC Curve

In [ ]:
vae_fpr, vae_tpr, _ = compute_roc_curve(dm.predict_labels, vae_scores)
fig = plot_roc_curve(vae_fpr, vae_tpr, auc=vae_metrics["roc_auc"], title="VAE ROC Curve")
fig.savefig(vae_plots_dir / "roc_curve.png", dpi=150, bbox_inches="tight")

### SIC Curve

Significance Improvement Characteristic: signal efficiency divided by the square root of background efficiency.

In [ ]:
vae_sic = compute_sic_curve(vae_fpr, vae_tpr)
fig = plot_sic_curve(vae_fpr, vae_tpr, vae_sic, title="VAE Significance Improvement")
fig.savefig(vae_plots_dir / "sic_curve.png", dpi=150, bbox_inches="tight")

### Per-Origin ROC

ROC AUC computed separately for each signal mass point.

In [ ]:
if vae_metrics.get("roc_per_origin"):
    fig = plot_roc_per_origin(
        vae_metrics["roc_per_origin"], title="VAE ROC AUC per Signal Origin"
    )
    fig.savefig(vae_plots_dir / "roc_per_origin.png", dpi=150, bbox_inches="tight")
    for origin, auc in sorted(vae_metrics["roc_per_origin"].items()):
        print(f"  {origin}: {auc:.4f}")

### Per-Feature Importance

Mean squared reconstruction error per feature, identifying which features contribute most to the anomaly score.

In [ ]:
vae_feat_errors = per_feature_error(x_orig, vae_x_hat).numpy()
vae_mean_feat_errors = vae_feat_errors.mean(axis=0)
fig = plot_per_feature_importance(
    vae_mean_feat_errors, dm.feature_names_, title="VAE Per-Feature Reconstruction Error"
)
fig.savefig(vae_plots_dir / "per_feature_importance.png", dpi=150, bbox_inches="tight")

## VAE Latent Space

### Latent Histograms

Per-dimension histograms of the latent means coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_histograms

fig = plot_latent_histograms(mu_np, labels=dm.predict_labels)
fig.savefig(vae_plots_dir / "latent_histograms.png", dpi=150, bbox_inches="tight")

### t-SNE

2D t-SNE embedding of the latent means coloured by background vs signal.

In [ ]:
from src.models.latent import compute_tsne
from src.models.plots import plot_latent_space_2d

max_tsne = 10_000
if len(mu_np) > max_tsne:
    rng = np.random.default_rng(cfg.seed)
    tsne_idx = rng.choice(len(mu_np), max_tsne, replace=False)
    tsne_mu = mu_np[tsne_idx]
    tsne_labels = dm.predict_labels[tsne_idx]
else:
    tsne_mu = mu_np
    tsne_labels = dm.predict_labels

embedding = compute_tsne(tsne_mu, seed=cfg.seed)
fig = plot_latent_space_2d(embedding, tsne_labels, method="t-SNE")
fig.savefig(vae_plots_dir / "tsne.png", dpi=150, bbox_inches="tight")

### KL per Dimension

Mean KL divergence per latent dimension. Identifies which dimensions encode the most information.

In [ ]:
from src.models.latent import compute_kl_per_dimension
from src.models.plots import plot_kl_per_dimension

kl_per_dim = compute_kl_per_dimension(mu_np, logvar_np)
fig = plot_kl_per_dimension(kl_per_dim)
fig.savefig(vae_plots_dir / "kl_per_dim.png", dpi=150, bbox_inches="tight")

### Latent Mean Spread

Variance of `mu` per latent dimension. Dimensions with variance below 0.1 indicate potential posterior collapse.

In [ ]:
from src.models.plots import plot_latent_mean_spread

fig = plot_latent_mean_spread(mu_np)
fig.savefig(vae_plots_dir / "mu_spread.png", dpi=150, bbox_inches="tight")

### Log-Variance Spread

Mean `logvar` per latent dimension. Very negative values may indicate posterior collapse.

In [ ]:
from src.models.plots import plot_logvar_spread

fig = plot_logvar_spread(logvar_np)
fig.savefig(vae_plots_dir / "logvar_spread.png", dpi=150, bbox_inches="tight")

### Mu vs Logvar

Scatter plot of mean `mu` vs mean `logvar` per latent dimension.

In [ ]:
from src.models.plots import plot_mu_vs_logvar

fig = plot_mu_vs_logvar(mu_np, logvar_np)
fig.savefig(vae_plots_dir / "mu_vs_logvar.png", dpi=150, bbox_inches="tight")

## Comparison

### AE vs VAE Reconstruction Error

Side-by-side comparison of reconstruction error distributions for both models.

In [ ]:
from src.models.plots import plot_reconstruction_comparison

fig = plot_reconstruction_comparison(ae_scores, vae_scores, dm.predict_labels)
fig.savefig(ae_plots_dir / "ae_vs_vae_comparison.png", dpi=150, bbox_inches="tight")

### Metrics Summary

Side-by-side comparison of evaluation metrics.

In [ ]:
print(f"{'Metric':<20} {'AE':>12} {'VAE':>12}")
print("-" * 46)
print(f"{'ROC AUC':<20} {ae_metrics['roc_auc']:>12.4f} {vae_metrics['roc_auc']:>12.4f}")
print(f"{'Max SIC':<20} {ae_metrics['max_sic']:>12.4f} {vae_metrics['max_sic']:>12.4f}")
print(f"{'Threshold':<20} {ae_metrics['threshold']:>12.6f} {vae_metrics['threshold']:>12.6f}")

## Save

### Scores DataFrames

Saving AE and VAE anomaly scores to parquet.

In [ ]:
ae_scores_path = dataframes_dir / "ae_scores.parquet"
vae_scores_path = dataframes_dir / "vae_scores.parquet"

ae_scores_df.to_parquet(ae_scores_path)
vae_scores_df.to_parquet(vae_scores_path)

print(f"Saved AE scores:  {ae_scores_path}")
print(f"Saved VAE scores: {vae_scores_path}")

### Metrics JSON

Saving evaluation metrics for both models.

In [ ]:
import json

ae_metrics_path = dataframes_dir / "ae_metrics.json"
vae_metrics_path = dataframes_dir / "vae_metrics.json"

with open(ae_metrics_path, "w") as f:
    json.dump(ae_metrics, f, indent=2)

with open(vae_metrics_path, "w") as f:
    json.dump(vae_metrics, f, indent=2)

print(f"Saved AE metrics:  {ae_metrics_path}")
print(f"Saved VAE metrics: {vae_metrics_path}")

### Finish WandB

Closing any active WandB run.

In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()